In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

forecast = pd.read_csv(
    project_dir / "data/processed/pm25_forecasting.csv",
    parse_dates=["date", "target_date"]
)

weather = pd.read_csv(
    project_dir / "data/processed/statewide_weather_daily.csv",
    parse_dates=["date"]
)

transport = pd.read_csv(
    project_dir / "data/processed/transport_features_daily.csv",
    parse_dates=["date"]
)

weather_cols = [
    "station_id",
    "date",
    "temperature_mean",
    "humidity_mean",
    "wind_speed_mean",
    "precipitation_total",
    "wind_sin_mean",
    "wind_cos_mean"
]

data = forecast.merge(
    weather[weather_cols],
    on=["station_id", "date"],
    how="left",
    validate="one_to_one"
)

data = data.merge(
    transport,
    on=["station_id", "date"],
    how="left",
    validate="one_to_one"
)

print("Combined records:", len(data))
print("Missing transport features:",
      data["upwind_frp_100km"].isna().sum())


Combined records: 8283
Missing transport features: 0


In [2]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

weather_features = [
    "temperature_mean",
    "humidity_mean",
    "wind_speed_mean",
    "precipitation_total",
    "wind_sin_mean",
    "wind_cos_mean"
]

fire_features = [
    "fire_count_50km",
    "fire_count_100km",
    "frp_sum_100km"
]

base_features = ["pm25"] + weather_features

original_features = base_features + fire_features

transport_features = (
    original_features + ["upwind_frp_100km"]
)

# Identical observations for each model
data = data.dropna(
    subset=transport_features + ["target_pm25"]
).copy()

train = data[data["target_date"] < "2025-09-01"]
test = data[data["target_date"] >= "2025-09-01"]

y_train = train["target_pm25"]
y_test = test["target_pm25"]

def fit_model(features):
    model = RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
    model.fit(train[features], y_train)
    return model.predict(test[features])

pred_weather_only = fit_model(base_features)
pred_original = fit_model(original_features)
pred_transport = fit_model(transport_features)

def evaluate(name, pred):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred))
    }

results = pd.DataFrame([
    evaluate("PM2.5 + Weather", pred_weather_only),
    evaluate("Fire + Weather", pred_original),
    evaluate("Fire + Weather + Transport", pred_transport)
])

display(results.round(4))


,Model,MAE,RMSE
0,PM2.5 + Weather,2.4232,11.3334
1,Fire + Weather,2.4107,11.3440
2,Fire + Weather + Transport,2.4041,11.3104
